In [0]:
%sql

DROP TABLE IF EXISTS medallion.default.employee_detail_silver;

CREATE TABLE medallion.default.employee_detail_silver AS 
SELECT 
  id,
  firstname,
  country,
  role,
  CASE WHEN role = 'Manager' OR role = 'Director' THEN 'Senior' ELSE 'Junior' END AS joblevel,
  current_date() AS lastupdate
FROM medallion.default.employee_detail_bronze;

In [0]:
%sql
SELECT * FROM medallion.default.employee_detail_silver;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW temp_view_total_roles AS
SELECT role, COUNT(*) AS total FROM medallion.default.employee_detail_silver GROUP BY role;
    
SELECT * FROM temp_view_total_roles 

-- TEMP VIEWS are not stored in unity catalog.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW temp_view_total_joblevel AS
SELECT joblevel, COUNT(*) AS total FROM medallion.default.employee_detail_silver GROUP BY joblevel;
    
SELECT * FROM temp_view_total_joblevel

In [0]:
%sql

DROP TABLE IF EXISTS medallion.default.role_info_gold;

CREATE TABLE IF NOT EXISTS medallion.default.role_info_gold(
  role STRING,
  total INT
);

DROP TABLE IF EXISTS medallion.default.joblevel_info_gold;

CREATE TABLE IF NOT EXISTS medallion.default.joblevel_info_gold(
  joblevel STRING,
  total INT
);

In [0]:
%sql
-- Databricks provides the INSERT OVERWRITE for adding new data by deleting existing data.
INSERT OVERWRITE medallion.default.role_info_gold
SELECT * FROM temp_view_total_roles;
    
INSERT OVERWRITE medallion.default.joblevel_info_gold
SELECT * FROM temp_view_total_joblevel;

In [0]:
%sql
SELECT * FROM medallion.default.role_info_gold;

In [0]:
%sql
SELECT * FROM medallion.default.joblevel_info_gold;